# Bibliotheken und Einstellungen
Diese Zelle zuerst ausführen. Alle Imports stehen gesammelt am Anfang.

In [ ]:
# Kleine Anzeigehilfe: Name links, blauer Balken, Prozentzahl rechts.
from IPython.display import display, HTML
from html import escape

def show_progress(name, completed, total, bar=None):
    percent = round(100 * completed / total)
    progress_html = (
        '<div style="display:flex;align-items:center;gap:12px;margin:8px 0;">'
        f'<span style="min-width:180px;">{escape(name)}</span>'
        '<div style="flex:1;background:#dbe4ed;border-radius:6px;height:16px;">'
        f'<div style="width:{percent}%;background:#2196f3;height:16px;'
        'border-radius:6px;"></div></div>'
        f'<span style="min-width:48px;text-align:right;">{percent}%</span></div>'
    )
    if bar is None:
        return display(HTML(progress_html), display_id=True)
    bar.update(HTML(progress_html))
    return bar

# Alle Imports für den geplanten Workflow: sechs Gruppen.
import_bar = show_progress("Bibliotheken laden", 0, 6)
from pathlib import Path
import time
show_progress("Standardbibliotheken", 1, 6, import_bar)

import joblib
import numpy as np
import pandas as pd
show_progress("Tabellenbibliotheken", 2, 6, import_bar)
import matplotlib.pyplot as plt
import seaborn as sns
show_progress("Grafikbibliotheken", 3, 6, import_bar)
from scipy.stats import randint, loguniform, uniform
show_progress("SciPy", 4, 6, import_bar)

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
show_progress("Vorverarbeitungsbibliotheken", 5, 6, import_bar)
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate,
    GridSearchCV,
    RandomizedSearchCV,
)
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")
show_progress("Bibliotheken geladen", 6, 6, import_bar)
print("Imports abgeschlossen. Noch kein Training durchgeführt.")


# CSV einfach laden

In [ ]:
# Zum Laden einer anderen CSV nur diesen Pfad ändern.
CSV_PATH = Path(
    "C:/Users/mpero/Desktop/Machine Learning/Supervised/Projekt/"
    "VSCode/data/big-five-personality/data/data_clean.csv"
)

if not CSV_PATH.is_file():
    raise FileNotFoundError(f"CSV nicht gefunden. Bitte CSV_PATH prüfen: {CSV_PATH}")

load_bar = show_progress("CSV laden", 0, 1)
df = pd.read_csv(CSV_PATH)
show_progress("CSV geladen", 1, 1, load_bar)

print("Geladene Datei:", CSV_PATH.name)
print(f"Dimensionen: {df.shape[0]:,} Zeilen, {df.shape[1]} Spalten")
print("Spalten:", df.columns.tolist())
display(df.head(5))


# Definition von X und y
target wird ausschließlich als Zielvariable verwendet. hand bleibt in X enthalten. Es werden keine Werte verändert oder Vorverarbeitungsschritte trainiert.

In [ ]:
# Eingabemerkmale und Zielvariable trennen; hand bleibt enthalten.
feature_bar = show_progress("Spalten prüfen", 0, 3)
required_columns = {
    "N6", "N8", "N7", "N1", "N9", "N10", "N3", "N5",
    "E3", "N2", "E5", "N4", "E7", "E4", "age", "C4",
    "E1", "A4", "E10", "E9", "gender", "hand", "target",
}
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(f"Benötigte Spalten fehlen: {sorted(missing_columns)}")
if df["target"].isna().any():
    raise ValueError("target enthält fehlende Werte. Bitte vor dem Training klären.")
show_progress("Spalten geprüft", 1, 3, feature_bar)

X = df.drop(columns=["target"]).copy()
y = df["target"].copy()
show_progress("X und y getrennt", 2, 3, feature_bar)

# gender und hand sind Kategorien, auch wenn sie als Zahlen codiert wären.
categorical_features = [
    column for column in X.columns
    if column in {"gender", "hand"}
    or not pd.api.types.is_numeric_dtype(X[column])
]
numeric_features = [
    column for column in X.columns if column not in categorical_features
]
show_progress("Merkmale bestimmt", 3, 3, feature_bar)

print("Numerische Merkmale:", numeric_features)
print("Kategoriale Merkmale:", categorical_features)
print("Target: target")


# Vorverarbeitung definieren
Numerische Merkmale: fehlende Werte werden mit dem Median ersetzt und standardisiert. 

Kategorien: der häufigste Wert wird einsetzt und One-Hot-Encoding verwendet. 

In [ ]:
# Hier werden nur die Schritte definiert, noch keine Werte gelernt.
preprocessing_bar = show_progress("Vorverarbeitung definieren", 0, 1)
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features),
], remainder="drop", sparse_threshold=0)
show_progress("Vorverarbeitung definiert", 1, 1, preprocessing_bar)

print("Vorverarbeitung definiert. Noch kein fit ausgeführt.")


# Vier Modell-Pipelines erstellen
Jede Pipeline verbindet Vorverarbeitung und Modell. Jede eigene Kopie verhindert, dass die Pipelines einen gemeinsam trainierten Preprocessor teilen. Welches Modell gewinnt, entscheiden später die CV-Ergebnisse.

In [ ]:
# Jede Pipeline erhält eine eigene Kopie der Vorverarbeitung.
pipeline_bar = show_progress("Pipelines erstellen", 0, 4)
pipe_logreg = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("model", LogisticRegression(max_iter=2000, random_state=42)),
])
show_progress("LogisticRegression erstellt", 1, 4, pipeline_bar)

pipe_knn = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("model", KNeighborsClassifier()),
])
show_progress("KNN erstellt", 2, 4, pipeline_bar)

pipe_rf = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("model", RandomForestClassifier(random_state=42)),
])
show_progress("RandomForest erstellt", 3, 4, pipeline_bar)

pipe_hgb = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("model", HistGradientBoostingClassifier(random_state=42)),
])
show_progress("HistGradientBoosting erstellt", 4, 4, pipeline_bar)

# Gemeinsame Sammlung für späteren Vergleich und Hyperparameter-Suche.
pipelines = {
    "pipe_logreg": pipe_logreg,
    "pipe_knn": pipe_knn,
    "pipe_rf": pipe_rf,
    "pipe_hgb": pipe_hgb,
}

pipeline_overview = pd.DataFrame([
    {
        "Pipeline": "pipe_logreg",
        "Modell": "LogisticRegression",
        "Familie": "Lineares Modell",
        "Warum?": "Einfacher, gut interpretierbarer Ausgangspunkt.",
    },
    {
        "Pipeline": "pipe_knn",
        "Modell": "KNeighborsClassifier",
        "Familie": "Distanzbasiert",
        "Warum?": "Klassifiziert anhand ähnlicher Antworten; Skalierung ist wichtig.",
    },
    {
        "Pipeline": "pipe_rf",
        "Modell": "RandomForestClassifier",
        "Familie": "Bagging / Entscheidungsbäume",
        "Warum?": "Erfasst nichtlineare Zusammenhänge und Merkmalskombinationen.",
    },
    {
        "Pipeline": "pipe_hgb",
        "Modell": "HistGradientBoostingClassifier",
        "Familie": "Boosting / Entscheidungsbäume",
        "Warum?": "Histogrammbasiertes Boosting als weiterer Kandidat für Tabellendaten.",
    },
])

# HTML-Anzeige mit Zeilenumbrüchen, ohne Pandas-Styler.
table_css = '''
<style>
table.pipeline-table { width: 100%; table-layout: fixed; border-collapse: collapse; }
table.pipeline-table th, table.pipeline-table td {
    white-space: normal; overflow-wrap: anywhere;
    text-align: left; vertical-align: top; padding: 8px;
}
table.pipeline-table th:nth-child(1) { width: 15%; }
table.pipeline-table th:nth-child(2) { width: 25%; }
table.pipeline-table th:nth-child(3) { width: 20%; }
table.pipeline-table th:nth-child(4) { width: 40%; }
</style>
'''

table_html = pipeline_overview.to_html(index=False, classes="pipeline-table", border=0)
display(HTML(table_css + table_html))
print("Vier Pipelines erstellt. Noch kein Training und noch kein Gewinner.")


# Train-Test-Split
Wir reservieren 20 % der Daten als Testset. test_size=0.20 bestimmt diesen Anteil, stratify=y erhält ungefähr dieselben Klassenanteile. Alle vier Modelle verwenden dieselbe Aufteilung. Die Testdaten bleiben bis zur finalen Bewertung unberührt.

In [ ]:
split_bar = show_progress("Train-Test-Split", 0, 1)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)
show_progress("Train-Test-Split abgeschlossen", 1, 1, split_bar)

print("Trainingsdaten:", X_train.shape)
print("Testdaten:", X_test.shape)


# Cross-Validation definieren
n_splits=5 legt fünf Durchläufe fest. shuffle=True mischt die Zeilen vor der Aufteilung; random_state=42 macht diese wiederholbar. StratifiedKFold berücksichtigt die Klassenanteile.

In [ ]:
cv_bar = show_progress("CV-Aufteilung definieren", 0, 1)
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)
show_progress("CV-Aufteilung definiert", 1, 1, cv_bar)


# Die Vier Pipelines trainieren und vergleichen
Die Schleife führt für jede Pipeline fünf CV-Trainings auf X_train und y_train aus: insgesamt 20 Trainingsläufe. Die Vorverarbeitung wird in jedem Durchlauf nur auf dessen Trainingsanteil gelernt.

Train F1-Macro: Wie gut erkennt das Modell die Klassen bei den Daten, mit denen es gelernt hat?

CV F1-Macro: Wie gut erkennt es die Klassen bei Daten, die es im jeweiligen Durchlauf noch nicht gesehen hat? Der Durchschnitt aus fünf Durchläufen ist unsere wichtigste Vergleichszahl.

CV F1-Streuung: Wie unterschiedlich sind die Ergebnisse der fünf Durchläufe? Eine kleine Streuung bedeutet gleichmäßigere Ergebnisse.

CV Accuracy: Welcher Anteil der Vorhersagen war richtig? 0,80 bedeutet beispielsweise 80 % richtige Vorhersagen.

In [ ]:
from IPython.display import display, HTML

def show_results_table(
    data, highlight_best=False, decimals=4, highlight_times=False
):
    table = data.round(decimals).to_html(
        index=False, border=0, classes="results-table"
    )

    before, separator, body = table.partition("<tbody>")
    rows = body.split("<tr>")

    for number in range(1, len(rows)):
        row_style = ""

        if highlight_times:
            times = data.iloc[:, -1]
            value = times.iloc[number - 1]

            if value == times.min():
                row_style = "background:#153b60;color:white;"
            elif value == times.max():
                row_style = "background:#7f1d1d;color:white;"

        elif highlight_best and number == 1:
            row_style = "background:#153b60;color:white;"

        if row_style:
            row_style += "font-weight:bold;font-style:italic;"

        rows[number] = f'<tr style="{row_style}">' + rows[number]

    table = before + separator + "".join(rows)

    style = """
    <style>
    table.results-table {
        font-size: 16px;
        border-collapse: collapse;
    }
    table.results-table th,
    table.results-table td {
        padding: 10px 14px;
        text-align: left;
    }
    </style>
    """

    display(HTML(style + table))

In [ ]:
results = []
timing_results = []

model_names = {
    "pipe_logreg": "LogisticRegression",
    "pipe_knn": "KNN",
    "pipe_rf": "RandomForest",
    "pipe_hgb": "HistGradientBoosting"
}

for name, pipeline in pipelines.items():
    model_name = model_names[name]
    model_bar = show_progress(f"CV: {model_name}", 0, cv.get_n_splits())

    start = time.time()

    # Jeden CV-Durchlauf einzeln ausführen, um echten Fortschritt zu zeigen.
    scores = {"train_f1_macro": [], "test_f1_macro": [], "test_accuracy": []}
    for fold, (train_indices, validation_indices) in enumerate(
        cv.split(X_train, y_train), start=1
    ):
        fold_scores = cross_validate(
            pipeline,
            X_train,
            y_train,
            cv=[(train_indices, validation_indices)],
            scoring={"f1_macro": "f1_macro", "accuracy": "accuracy"},
            return_train_score=True,
            error_score="raise"
        )
        for metric in scores:
            scores[metric].append(fold_scores[metric][0])
        show_progress(f"CV: {model_name}", fold, cv.get_n_splits(), model_bar)

    # Arrays ermöglichen dieselbe Mittelwert-/Streuungsberechnung wie zuvor.
    scores = {metric: np.array(values) for metric, values in scores.items()}

    duration = time.time() - start
    timing_results.append({
        "Modell": model_name,
        "CV-Dauer (Sekunden)": duration
    })

    results.append({
        "Modell": model_name,
        "Train F1-Macro": scores["train_f1_macro"].mean(),
        "CV F1-Macro": scores["test_f1_macro"].mean(),
        "CV F1-Streuung": scores["test_f1_macro"].std(),
        "CV Accuracy": scores["test_accuracy"].mean()
    })

results_df = pd.DataFrame(results)
results_df = results_df.sort_values("CV F1-Macro", ascending=False).reset_index(drop=True)
# display(HTML(results_df.round(4).to_html(index=False, border=0)))
show_results_table(results_df, highlight_best=True)


# Zeit für alle fünf CV-Durchläufe je Pipeline, inklusive Vorverarbeitung.
timing_df = pd.DataFrame(timing_results)
print("Laufzeiten der Cross-Validation:")
#display(HTML(timing_df.round(2).to_html(index=False, border=0)))
show_results_table(timing_df, decimals=2, highlight_times=True)
